# 第 09 天：质量因子 2

> 来自《30 天因子研究计划》第 9 天  
> 主题：质量因子 2  
> 必做：净利润增长率  
> 选做：现金流质量  
> 目标产出：质量因子报告

---

## 0. 今天你要真正学会什么？

第 8 天我们学习了 ROE、ROA、毛利率，它们偏静态地描述公司当前赚钱能力。  
今天继续质量因子，但加入两个更有穿透力的问题：


利润有没有增长？
利润有没有现金流支撑？


今天重点学习：

1. 净利润增长率怎么计算。
2. 为什么增长率容易被低基数和一次性因素扭曲。
3. 现金流质量是什么。
4. 为什么“有利润但没现金流”要警惕。
5. 如何输出一份质量因子报告。

一句话版：

> 好公司不仅要会赚钱，还要利润增长健康、现金流跟得上。

---

## 1. 先建立直觉：利润和现金不是一回事

一家公司利润表上赚了 10 亿，不代表现金账户真的多了 10 亿。

原因可能包括：

- 应收账款增加：卖出去了但钱还没收回来。
- 存货增加：利润看起来不错，但货压在仓库。
- 会计估计：收入确认和费用摊销有主观成分。
- 一次性收益：卖资产带来利润，但不可持续。

所以质量因子不能只看利润高不高，还要看：


利润是否增长？
利润是否变成现金？
增长是否稳定？


---

## 2. 两个核心指标

### 2.1 净利润增长率

公式：


净利润增长率 = 本期净利润 / 上期净利润 - 1


如果去年净利润 10 亿，今年 13 亿：


增长率 = 13 / 10 - 1 = 30%


风险点：

- 去年利润很低时，增长率会虚高。
- 去年亏损、今年盈利时，增长率公式可能失真。
- 一次性收益会抬高增长率。

### 2.2 现金流质量

常见定义：


现金流质量 = 经营现金流 / 净利润


直觉：

> 净利润里有多少变成了经营现金流。

如果净利润 10 亿，经营现金流 12 亿，现金流质量 = 1.2，通常不错。  
如果净利润 10 亿，经营现金流 1 亿，现金流质量 = 0.1，就要追问利润质量。

---

## 3. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260705)


如果缺包，可以先安装：


In [ ]:
pip install numpy pandas matplotlib


---

## 4. 构造两期财务数据

我们模拟上一期和本期净利润、经营现金流、收入等数据。


In [ ]:
n = 600

base_income = rng.lognormal(mean=6.8, sigma=0.9, size=n)
growth_driver = rng.normal(loc=0.08, scale=0.22, size=n)

df = pd.DataFrame({
    "ticker": [f"Stock_{i:03d}" for i in range(n)],
    "net_income_prev": base_income,
})

df["net_income_curr"] = df["net_income_prev"] * (1 + growth_driver)

# 少量亏损或低基数样本
df.loc[rng.choice(n, size=35, replace=False), "net_income_prev"] *= -0.4
df.loc[rng.choice(n, size=25, replace=False), "net_income_curr"] *= -0.5

# 经营现金流：大体跟随净利润，但加入噪声
cash_quality_true = np.clip(rng.normal(loc=1.05, scale=0.35, size=n), -0.2, 2.2)
df["operating_cash_flow"] = df["net_income_curr"] * cash_quality_true + rng.normal(0, 120, size=n)

# 加入收入和毛利率，方便报告更完整
df["revenue"] = rng.lognormal(mean=8.6, sigma=0.8, size=n)
df["gross_profit"] = df["revenue"] * np.clip(rng.normal(0.35, 0.13, size=n), 0.03, 0.85)

df.head()


---

## 5. 计算净利润增长率

直接公式：


In [ ]:
df["net_profit_growth_raw"] = df["net_income_curr"] / df["net_income_prev"] - 1
df[["ticker", "net_income_prev", "net_income_curr", "net_profit_growth_raw"]].head()


但当上一期净利润为负或接近 0 时，这个指标会非常危险。

我们做一个更保守版本：


In [ ]:
min_base = df["net_income_prev"].abs().quantile(0.10)

df["net_profit_growth"] = np.where(
    df["net_income_prev"] > min_base,
    df["net_income_curr"] / df["net_income_prev"] - 1,
    np.nan
)

df[["net_profit_growth_raw", "net_profit_growth"]].describe()


解释：

- 如果上一期利润太小或为负，增长率设为空。
- 这不是唯一处理方式，但比直接排序更稳。

---

## 6. 计算现金流质量


In [ ]:
df["cash_flow_quality_raw"] = df["operating_cash_flow"] / df["net_income_curr"]

df["cash_flow_quality"] = np.where(
    df["net_income_curr"] > 0,
    df["cash_flow_quality_raw"],
    np.nan
)

df[["ticker", "net_income_curr", "operating_cash_flow", "cash_flow_quality"]].head()


描述统计：


In [ ]:
df[["cash_flow_quality"]].describe()


现金流质量的粗略直觉：

| 数值 | 可能含义 |
| ---: | --- |
| < 0 | 盈利但经营现金流为负，需警惕 |
| 0 - 0.5 | 利润现金含量弱 |
| 0.5 - 1.0 | 尚可 |
| 1.0 附近或以上 | 利润现金支持较好 |
| 过高 | 可能有营运资本释放或一次性因素 |

---

## 7. 构造质量增长因子

### 7.1 工具函数


In [ ]:
def winsorize_series(s: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)


def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()


### 7.2 因子标准化

净利润增长率越高越好，现金流质量越高通常越好，但过高也需要解释。这里先按正向处理。


In [ ]:
df["quality_growth"] = zscore(winsorize_series(df["net_profit_growth"]))
df["quality_cash_flow"] = zscore(winsorize_series(df["cash_flow_quality"]))
df["gross_margin"] = df["gross_profit"] / df["revenue"]
df["quality_gross_margin"] = zscore(winsorize_series(df["gross_margin"]))

quality_cols = ["quality_growth", "quality_cash_flow", "quality_gross_margin"]
df["quality_score"] = df[quality_cols].mean(axis=1, skipna=True)

df[["ticker", "net_profit_growth", "cash_flow_quality", "gross_margin", "quality_score"]].head()


---

## 8. 识别“有利润但没现金流”的公司


In [ ]:
warning = df[
    (df["net_income_curr"] > 0) &
    (df["cash_flow_quality"] < 0.3)
].copy()

warning[["ticker", "net_income_curr", "operating_cash_flow", "cash_flow_quality"]].head(10)


这类公司不一定都差，但值得进一步研究：

- 是否应收账款大增？
- 是否存货增加？
- 是否收入确认激进？
- 是否行业季节性导致？

---

## 9. 质量因子报告：分布和相关性


In [ ]:
report_cols = ["quality_growth", "quality_cash_flow", "quality_gross_margin", "quality_score"]
df[report_cols].describe()


In [ ]:
corr = df[report_cols].corr()
corr


画热力图：


In [ ]:
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(report_cols)), report_cols, rotation=45)
plt.yticks(range(len(report_cols)), report_cols)
plt.title("Quality Factor Correlation")
plt.tight_layout()
plt.show()


如果增长和现金流质量相关性不高，说明它们提供了不同维度的信息。

---

## 10. 模拟未来收益并做分组报告


In [ ]:
noise = rng.normal(0, 0.055, size=n)
df["future_20d_ret"] = 0.010 * df["quality_score"].fillna(0) + noise

rank_ic = df["quality_score"].corr(df["future_20d_ret"], method="spearman")
print("质量综合分数 Rank IC:", round(rank_ic, 4))


分组报告：


In [ ]:
valid = df.dropna(subset=["quality_score", "future_20d_ret"]).copy()
valid["group"] = pd.qcut(
    valid["quality_score"].rank(method="first"),
    q=5,
    labels=["G1 低质量", "G2", "G3", "G4", "G5 高质量"]
)

group_report = valid.groupby("group", observed=True).agg(
    stock_count=("ticker", "count"),
    avg_quality_score=("quality_score", "mean"),
    avg_growth=("net_profit_growth", "mean"),
    avg_cash_flow_quality=("cash_flow_quality", "mean"),
    avg_gross_margin=("gross_margin", "mean"),
    avg_future_20d_ret=("future_20d_ret", "mean"),
)

group_report


画图：


In [ ]:
group_report["avg_future_20d_ret"].plot(kind="bar", title="质量分组未来 20 日平均收益")
plt.ylabel("Future 20D Return")
plt.xticks(rotation=30)
plt.show()


---

## 11. 今日目标产出：质量因子报告


In [ ]:
def build_quality_report(data: pd.DataFrame) -> dict:
    factor_cols = ["quality_growth", "quality_cash_flow", "quality_gross_margin", "quality_score"]
    valid = data.dropna(subset=["quality_score", "future_20d_ret"]).copy()
    valid["group"] = pd.qcut(
        valid["quality_score"].rank(method="first"),
        q=5,
        labels=["G1 低质量", "G2", "G3", "G4", "G5 高质量"]
    )

    factor_summary = data[factor_cols].describe()
    factor_corr = data[factor_cols].corr()
    rank_ic = data["quality_score"].corr(data["future_20d_ret"], method="spearman")
    group = valid.groupby("group", observed=True).agg(
        stock_count=("ticker", "count"),
        avg_quality_score=("quality_score", "mean"),
        avg_growth=("net_profit_growth", "mean"),
        avg_cash_flow_quality=("cash_flow_quality", "mean"),
        avg_gross_margin=("gross_margin", "mean"),
        avg_future_20d_ret=("future_20d_ret", "mean"),
    )
    warnings = data[
        (data["net_income_curr"] > 0) &
        (data["cash_flow_quality"] < 0.3)
    ][["ticker", "net_income_curr", "operating_cash_flow", "cash_flow_quality"]]

    return {
        "factor_summary": factor_summary,
        "factor_corr": factor_corr,
        "rank_ic": rank_ic,
        "group_report": group,
        "cash_flow_warnings": warnings,
    }


quality_report = build_quality_report(df)
quality_report["rank_ic"], quality_report["group_report"]


现金流预警样本：


In [ ]:
quality_report["cash_flow_warnings"].head()


这就是今天的目标产出：一份包含增长、现金流质量、分组表现和预警样本的质量因子报告。

---

## 12. 实战注意事项

### 12.1 增长率要防低基数

去年利润很低，今年稍微恢复，增长率可能夸张。

### 12.2 现金流质量要结合行业

某些行业有明显季节性，单期现金流可能波动很大。

### 12.3 利润增长要看持续性

单年高增长不如连续多年稳定增长。

### 12.4 现金流质量不是越高越无脑好

过高可能来自存货下降、应付款增加或一次性营运资本释放。

---

## 13. 今天的知识图谱


In [ ]:
mindmap
  root((质量因子2))
    净利润增长率
      本期净利润除以上期净利润减1
      看成长
      低基数风险
      亏损转正难处理
    现金流质量
      经营现金流除以净利润
      看利润含金量
      盈利无现金需警惕
      行业季节性
    毛利率补充
      定价权
      成本控制
    报告
      因子分布
      相关性
      RankIC
      分组收益
      预警样本
    风险点
      一次性收益
      应收账款
      低基数
      现金流季节性


---

## 14. 初学者最容易踩的 8 个坑

### 坑 1：直接排序所有增长率

低基数会制造非常夸张的增长率。

### 坑 2：亏损转正处理太粗糙

亏损转正很重要，但不能简单用普通增长率排序。

### 坑 3：忽略现金流

利润增长但现金流很差，要警惕利润质量。

### 坑 4：认为现金流质量越高越好

异常高也可能来自一次性营运资本变化。

### 坑 5：只看单期

质量和增长都要看持续性。

### 坑 6：跨行业硬比

现金流周期和利润率行业差异很大。

### 坑 7：忽略财务披露时点

基本面因子必须严格避免未来函数。

### 坑 8：把质量好等同于股票会涨

质量因子仍需估值、预期和市场环境配合。

---

## 15. 今天的动手作业

### 作业 A：解释指标

用自己的话解释：

1. 净利润增长率是什么？
2. 现金流质量是什么？
3. 为什么利润增长要结合现金流？

### 作业 B：运行质量报告

运行本文所有代码，输出：

- 因子描述统计
- 因子相关矩阵
- Rank IC
- 分组报告
- 现金流预警样本

### 作业 C：检查低现金流公司

查看 `cash_flow_quality < 0.3` 的公司，判断它们为什么需要预警。

### 作业 D：修改综合权重

把质量综合分数改成：


quality_score = 0.4 * quality_growth + 0.4 * quality_cash_flow + 0.2 * quality_gross_margin


观察分组表现变化。

---

## 16. 自测题

### 题 1

净利润增长率的公式是什么？

答案：本期净利润 / 上期净利润 - 1。

### 题 2

现金流质量的常见公式是什么？

答案：经营现金流 / 净利润。

### 题 3

为什么低基数会扭曲增长率？

答案：上一期利润很小时，本期小幅增加也会产生极高增长率。

### 题 4

盈利但经营现金流为负说明什么？

答案：利润现金含量较弱，需要检查应收账款、存货、收入确认和营运资本变化。

### 题 5

质量因子报告应包含哪些内容？

答案：因子分布、相关性、IC、分组收益和异常预警样本。

---

## 17. 今日复盘模板


第 09 天复盘：质量因子 2

1. 我今天理解的净利润增长率：

2. 我今天理解的现金流质量：

3. 我构建的质量报告字段：

4. 我发现的现金流预警样本：

5. 我认为增长率最大风险：

6. 我认为现金流质量最大风险：

7. 明天学习动量因子前，我需要准备：


---

## 18. 明天预告：动量因子 1

前几天我们学的是基本面因子：


价值：便宜不便宜
质量：公司好不好


明天开始进入价格行为因子：


动量：过去强的股票，未来是否继续强？


---

## 19. 一句话收尾

利润增长让公司看起来有活力，现金流质量让这种活力更可信。

> 真正好的质量因子，不只看利润表上的漂亮数字，也要看现金有没有跟上。

---

## 20. 仅供学习的提醒

本文所有示例使用模拟数据，仅用于解释质量因子报告方法，不构成任何投资建议。真实研究需要处理财务披露时点、会计口径、行业差异、一次性损益、现金流季节性和样本外验证。

---

# 统一高质量增强模块

> 本增强模块用于把第 09 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：质量因子2
- 必做：净利润增长率
- 选做：现金流质量
- 目标产出：质量因子报告

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

利润表说赚了钱，现金流量表却没看到现金流入，这种增长要谨慎。质量因子要同时看增长和现金含量。

这个例子背后的关键直觉是：

> 增长要健康，利润要有现金流支持。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


质量因子2
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 质量因子报告


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(109)
n = 350
prev = rng.lognormal(6.5, .9, n)
growth = rng.normal(.08, .25, n)
curr = prev * (1 + growth)
ocf = curr * np.clip(rng.normal(1.05, .35, n), -.2, 2.2)
df = pd.DataFrame({"ticker": [f"S{i:03d}" for i in range(n)], "prev_income": prev, "curr_income": curr, "ocf": ocf})
min_base = df["prev_income"].quantile(.1)
df["profit_growth"] = np.where(df["prev_income"] > min_base, df["curr_income"] / df["prev_income"] - 1, np.nan)
df["cash_quality"] = np.where(df["curr_income"] > 0, df["ocf"] / df["curr_income"], np.nan)

def zscore(s):
    s = s.clip(s.quantile(.01), s.quantile(.99))
    return (s - s.mean()) / s.std()

df["quality_growth"] = zscore(df["profit_growth"])
df["quality_cash"] = zscore(df["cash_quality"])
df["quality_score"] = df[["quality_growth", "quality_cash"]].mean(axis=1)
print(df[["profit_growth", "cash_quality", "quality_score"]].describe().round(3))


## E. 产出验收标准

完成今天课程后，你的 `质量因子报告` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `质量因子2` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `质量因子报告` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `质量因子报告`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 09 天复盘：质量因子2

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
